In [1]:
!pip install -U trl accelerate transformers bitsandbytes peft

In [2]:
!pip install PyMuPDF

In [3]:
from datasets import Dataset, load_dataset

**our costum data for domain specific fine tune**

In [4]:
import fitz

In [5]:
from google.colab import files
uploaded = files.upload()

Saving Rithik_Tiwari_Resume_Summary.pdf to Rithik_Tiwari_Resume_Summary.pdf


In [7]:
def extract_text_from_pdf(pdf_path):
  text_blocks= []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return text_blocks

In [8]:
pdf_texts = extract_text_from_pdf("/content/Rithik_Tiwari_Resume_Summary.pdf")
print(f"Successfully extracted {len(pdf_texts)} text blocks from the PDF.")

Successfully extracted 2 text blocks from the PDF.


In [9]:
pdf_texts

['Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Multi-Agent 

In [10]:
import re

def split_paragraphs(pages):
    paragraphs = []

    for page_text in pages:

        # Split text wherever there are blank lines
        chunks = re.split(r'\n\s*\n', page_text)

        for chunk in chunks:
            clean = chunk.strip()

            # Ignore very small text pieces
            if len(clean) > 30:
                paragraphs.append(clean)

    return paragraphs

In [11]:
paragraphs = split_paragraphs(pdf_texts)
paragraphs

['Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Multi-Agent 

In [12]:
data = [{"text": p}for p in paragraphs]
data

[{'text': 'Professional Profile & Career Overview\nRithik Tiwari | AI/ML Engineer\nProfessional Summary\nRithik Tiwari is an aspiring AI/ML Engineer specializing in production-ready machine learning systems, NLP pipelines,\nand intelligent multi-agent architectures. He has hands-on experience in developing scalable AI applications using\nPython, PyTorch, LangChain, Docker, and Airflow. His work demonstrates a strong focus on model optimization,\nautomation, MLOps workflows, and deploying efficient AI systems for real-world use cases.\nEducation\nChandigarh Engineering College, Mohali, Punjab\nBachelor of Technology (B.Tech) in Artificial Intelligence & Machine Learning (2022 – 2026)\nTechnical Skills\nCategory\nSkills\nProgramming\nPython, C++, R, SQL\nAI/ML\nMachine Learning, Deep Learning, NLP, Feature Engineering\nFrameworks\nPyTorch, TensorFlow, Scikit-learn, LangChain, Hugging Face\nMLOps\nMLflow, Airflow, Docker, Kafka, CI/CD Pipelines\nGenAI\nRAG Systems, Prompt Engineering, Mul

In [13]:
dataset = Dataset.from_list(data)


In [14]:
dataset

Dataset({
    features: ['text'],
    num_rows: 2
})

**LETS SELECT THE MODEL**

In [15]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [17]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [18]:
def tokenizer_fn(examples):
  tokens= tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
  tokens["labels"]=  tokens["input_ids"].copy()
  return tokens

In [21]:
tokenized = dataset.map(tokenizer_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [22]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

In [23]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [28]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps =500,
    save_total_limit= 2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to='None'
)

In [ ]:
from transformers import TrainingArguments
help(TrainingArguments)

# **now let use LORA method**

In [30]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [31]:
!pip install -U trl accelerate transformers bitsandbytes peft

In [32]:
from peft import LoraConfig, get_peft_model, TaskType